# Fase 01 : Limpieza y primera exploración de datos 
**Institución:** Universidad Católica Boliviana "San Pablo"  
**Programa:** Maestría en Ciencia de Datos  
**Autor:** Carlos Andrés Pérez Guzmán  
**Fecha:** Septiembre 2026  

---
**Entorno y Objetivos del Notebook**
> **Nota:** Limpieza, y selección de casos de interés 

* **Fase del Proyecto:** Primera fase
* **Core Tech Stack:** `Python` (Programación general para ciencia de datos)
---

<h3 style="color:#05856d">1. Librerías</h3>
<hr>

In [9]:
from datetime import date

import polars as pl
import polars.selectors as cs
import s_01_functions as fn_01
from pyprojroot import here

<h3 style="color:#05856d">2. Carga de información</h3>
<hr>

Se importan las funciones de carga y limpieza de datos

In [10]:
niveles_botellas = [
    '20', '25', '50', '100', '200', '375', '700', '750','800','850',
    '1000','1100','1125','1750', '3000', '3750', '4500' ]

work_folder = here()/'00_raw_data'/'2026'

df = (
    fn_01.load_data_own( file_folder_param = work_folder)

    # Formatear fecha y aplicar filtro
    .with_columns( pl.col('ordered_on').str.to_date(format = '%Y-%m-%d') )
    #.filter(pl.col('ordered_on').is_between(date(2026, 1, 1), date(2026, 5, 31)))
    .filter(pl.col('ordered_on').is_between(date(2026, 1, 1), date(2026, 8, 31)))

    # Limpiar texto
    .with_columns( fn_01.limpiar_texto_own(column_param = cs.string()) )

    # Asignar tipo de variables
    .with_columns( pl.col('im_desc','category_name','store_no').cast(dtype = pl.Categorical()),
                   pl.col('sales_bottles').cast(dtype = pl.Int64()),
                   pl.col('ordered_on').dt.strftime('%Y-%m').cast(dtype = pl.Categorical()).alias('period_ordered_on'))

    # Descartar valores negativos de "sales_bottles". Puede tratarse de una salida o devolución de producto
    .filter(pl.col('sales_bottles')>0) 
    
    # Eliminar decimales en el campo bottle_volume_ml y convertir en una variable categórica
    .with_columns( pl.col('bottle_volume_ml')
                    .str.replace(pattern = '\\.0$', value = '')
                    .cast(dtype = pl.Enum(niveles_botellas)) )

    # Ordenar variables
    .pipe(lambda l_frame: l_frame.select(sorted(l_frame.collect_schema().names())))

    )

df.collect_schema()

Iniciando carga de datos locales desde: g:\My Drive\01. Formación\02 Maestría en ciencia de datos UCB\Segunda fase\Proyecto_UCB_2da_fase\00_raw_data\2026


Schema([('bottle_volume_ml',
         Enum(categories=['20', '25', '50', '100', '200', '375', '700', '750', '800', '850', '1000', '1100', '1125', '1750', '3000', '3750', '4500'])),
        ('category_code', Int64),
        ('category_name', Categorical),
        ('county_fips_code', Int64),
        ('county_name', String),
        ('im_desc', Categorical),
        ('invoice_id', String),
        ('item_no', Int64),
        ('ordered_on', Date),
        ('pack', Int64),
        ('period_ordered_on', Categorical),
        ('sales_bottles', Int64),
        ('sales_dollars', Float64),
        ('sales_gallons', Float64),
        ('sales_liters', Float64),
        ('state_bottle_cost', Float64),
        ('state_bottle_retail', Float64),
        ('store_address', String),
        ('store_city', String),
        ('store_name', String),
        ('store_no', Categorical),
        ('store_zip_code', Int64),
        ('vendor_name', String),
        ('vendor_number', Int64)])

<h3 style="color:#05856d">3. Selección de variables y transformaciones particulares</h3>
<hr>

Es importante identificar qué variables tienen alguna funcionalidad estratégica al construir un modelo de decisión.

Se elimina la categoría del producto debido a que no es consistente con el número de producto (item_no)

In [11]:
# Variables con utilidad estratégica
df_selected_variables = [ 'ordered_on','item_no','im_desc','sales_bottles','store_no',
                          'category_name','sales_dollars','pack','bottle_volume_ml',
                          'period_ordered_on' ]

# Variables sin utilidad estratégica
df_eliminated_variables = set(df.collect_schema().names()) - set(df_selected_variables)
print('Variables sin utilidad estratégica:', *sorted(df_eliminated_variables), '\n', sep = '\n')

# Selección de variables de interés
df_01 = df.select(df_selected_variables).collect()
df_01.schema

Variables sin utilidad estratégica:
category_code
county_fips_code
county_name
invoice_id
sales_gallons
sales_liters
state_bottle_cost
state_bottle_retail
store_address
store_city
store_name
store_zip_code
vendor_name
vendor_number




Schema([('ordered_on', Date),
        ('item_no', Int64),
        ('im_desc', Categorical),
        ('sales_bottles', Int64),
        ('store_no', Categorical),
        ('category_name', Categorical),
        ('sales_dollars', Float64),
        ('pack', Int64),
        ('bottle_volume_ml',
         Enum(categories=['20', '25', '50', '100', '200', '375', '700', '750', '800', '850', '1000', '1100', '1125', '1750', '3000', '3750', '4500'])),
        ('period_ordered_on', Categorical)])

In [12]:
df_01.glimpse()

Rows: 1602961
Columns: 10
$ ordered_on        <date> 2026-01-01, 2026-01-01, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02
$ item_no            <i64> 88296, 69947, 87484, 88296, 84393, 101180, 36875, 36308, 36306, 36968
$ im_desc            <cat> PATRON SILVER, RUMPLE MINZE PEPPERMINT SCHNAPPS LIQUEUR, DON JULIO BLANCO, PATRON SILVER, 99 WATERMELON PET MINI, 99 FRUITS MINI, BLUE OX VODKA, HAWKEYE VODKA, HAWKEYE VODKA, NEW AMSTERDAM 80PRF
$ sales_bottles      <i64> 60, 36, 3, 12, 1, 1, 6, 6, 12, 4
$ store_no           <cat> 5444, 5444, 4417, 4417, 4417, 4417, 4417, 4417, 4417, 4417
$ category_name      <cat> 100% AGAVE TEQUILA, IMPORTED SCHNAPPS, 100% AGAVE TEQUILA, 100% AGAVE TEQUILA, AMERICAN CORDIALS & LIQUEUR, AMERICAN SCHNAPPS, AMERICAN VODKAS, AMERICAN VODKAS, AMERICAN VODKAS, AMERICAN VODKAS
$ sales_dollars      <f64> 2249.4, 917.64, 67.0, 449.0, 81.0, 81.0, 68.0, 67.0, 63.0, 24.0
$ pack               <i64> 12, 12, 12, 12, 1, 1, 6,

Revisar integridad de la llave del producto

In [13]:
duplicados_im_desc = (
    df_01
    .select('item_no', 'im_desc')
    .unique()
    .filter(pl.col('im_desc').is_duplicated())
    .sort('im_desc','item_no')
    )

print(f"Duplicados por la descripción del producto (im_desc): {duplicados_im_desc.height > 0}. Casos: {duplicados_im_desc.height}")
print(duplicados_im_desc.head())

duplicados_item_no = (
    df_01
    .select('item_no', 'im_desc')
    .unique()
    .filter(pl.col('item_no').is_duplicated())
    .sort('item_no','im_desc')
    )

print(f"Duplicados por número de ítem (item_no): {duplicados_item_no.height > 0}. Casos: {duplicados_item_no.height}")
print(duplicados_item_no.head())


duplicados_compuesto = (
    df_01
    .select('item_no','im_desc')
    .unique()
    .is_duplicated().any()
    )

print(f"\nDuplicados por la descripción del producto (im_desc) y número de item (item_no): {duplicados_compuesto}")


Duplicados por la descripción del producto (im_desc): True. Casos: 1170
shape: (5, 2)
┌─────────┬──────────────────────────┐
│ item_no ┆ im_desc                  │
│ ---     ┆ ---                      │
│ i64     ┆ cat                      │
╞═════════╪══════════════════════════╡
│ 21235   ┆ 1792 SMALL BATCH BOURBON │
│ 21239   ┆ 1792 SMALL BATCH BOURBON │
│ 64933   ┆ 1800 COCONUT             │
│ 964931  ┆ 1800 COCONUT             │
│ 89206   ┆ 1800 REPOSADO            │
└─────────┴──────────────────────────┘
Duplicados por número de ítem (item_no): True. Casos: 176
shape: (5, 2)
┌─────────┬─────────────────────────────────┐
│ item_no ┆ im_desc                         │
│ ---     ┆ ---                             │
│ i64     ┆ cat                             │
╞═════════╪═════════════════════════════════╡
│ 1799    ┆ CAPTAIN MORGAN ORIGINAL SPICED… │
│ 1799    ┆ CAPTAIN MORGAN ORIGINAL SPICED… │
│ 7224    ┆ MACALLAN SHERRY OAK 12YO 110 P… │
│ 7224    ┆ MACALLAN SHERRY OAK 12YO 110PF  │

Aplicar una llave consistente

In [14]:
df_01 = (
    df_01
    .with_columns(
        pl.format(
            'PRD_{}',
            pl.struct(['item_no', 'im_desc'])
            .rank(method = 'dense')
            .cast(pl.String)
            .str.zfill(6) ).alias('product_code'))
    .pipe(lambda df: df.select(sorted(df.columns)))
    )

df_01.glimpse()

Rows: 1602961
Columns: 11
$ bottle_volume_ml  <enum> 750, 1000, 375, 750, 50, 50, 1750, 1750, 750, 375
$ category_name      <cat> 100% AGAVE TEQUILA, IMPORTED SCHNAPPS, 100% AGAVE TEQUILA, 100% AGAVE TEQUILA, AMERICAN CORDIALS & LIQUEUR, AMERICAN SCHNAPPS, AMERICAN VODKAS, AMERICAN VODKAS, AMERICAN VODKAS, AMERICAN VODKAS
$ im_desc            <cat> PATRON SILVER, RUMPLE MINZE PEPPERMINT SCHNAPPS LIQUEUR, DON JULIO BLANCO, PATRON SILVER, 99 WATERMELON PET MINI, 99 FRUITS MINI, BLUE OX VODKA, HAWKEYE VODKA, HAWKEYE VODKA, NEW AMSTERDAM 80PRF
$ item_no            <i64> 88296, 69947, 87484, 88296, 84393, 101180, 36875, 36308, 36306, 36968
$ ordered_on        <date> 2026-01-01, 2026-01-01, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02, 2026-01-02
$ pack               <i64> 12, 12, 12, 12, 1, 1, 6, 6, 12, 24
$ period_ordered_on  <cat> 2026-01, 2026-01, 2026-01, 2026-01, 2026-01, 2026-01, 2026-01, 2026-01, 2026-01, 2026-01
$ product_code       <str> 'PRD_0

Revisar datos faltantes

In [15]:
null_variables = (
    df_01
    .select( 
        pl.all().null_count().name.suffix('_nulos'),
        (pl.all().null_count() / pl.len() * 100).name.suffix('_pct'))
    .unpivot()
    .with_columns(
        columna = pl.col('variable').str.replace('_(nulos|pct)$', ''),
        metrica = pl.col("variable").str.extract('_(nulos|pct)$', 1) )
    .pivot(index = 'columna', on = 'metrica', values = 'value')
    .sort('pct', descending = True)
    )

print(null_variables)

shape: (11, 3)
┌───────────────────┬───────┬──────────┐
│ columna           ┆ nulos ┆ pct      │
│ ---               ┆ ---   ┆ ---      │
│ str               ┆ f64   ┆ f64      │
╞═══════════════════╪═══════╪══════════╡
│ category_name     ┆ 26.0  ┆ 0.001622 │
│ bottle_volume_ml  ┆ 0.0   ┆ 0.0      │
│ im_desc           ┆ 0.0   ┆ 0.0      │
│ item_no           ┆ 0.0   ┆ 0.0      │
│ ordered_on        ┆ 0.0   ┆ 0.0      │
│ …                 ┆ …     ┆ …        │
│ period_ordered_on ┆ 0.0   ┆ 0.0      │
│ product_code      ┆ 0.0   ┆ 0.0      │
│ sales_bottles     ┆ 0.0   ┆ 0.0      │
│ sales_dollars     ┆ 0.0   ┆ 0.0      │
│ store_no          ┆ 0.0   ┆ 0.0      │
└───────────────────┴───────┴──────────┘


<h3 style="color:#05856d">4. Exportar datos limpios</h3>
<hr>

In [16]:
df_01.write_parquet( file = here()/'01_cleaned_data'/'df_01.parquet', 
                     compression = 'zstd',
                     compression_level = 12 )